#  核心算法与损失函数配置
这部分定义了模型如何学习，以及优化的目标是什么。

| 参数名 | 值/示例 | 含义解释 |
| :--- | :--- | :--- |
| `adv_estimator` | `grpo` | 优势估计算法。这里选择了 GRPO，一种不需要 Critic 模型、通过组内相对表现来计算优势的算法。 |
| `loss_mode` | `future_kl` | 策略损失模式。这是 FIPO 算法特有的，表示使用“未来 KL 散度”来计算 Loss，而非传统的 PPO Loss。 |
| `decay_rate` | `32.0` | 衰减率。用于 FIPO 算法中，控制未来 KL 散度权重的衰减速度。 |
| `chunk_size` | `128` | 分块大小。计算 Future KL 时分块处理的大小，用于节省显存。 |
| `future_kl_window` | `-1` | 窗口大小。计算 Future KL 时考虑的上下文窗口，`-1` 通常表示全局或默认。 |
| `use_kl_loss` | `False` | 是否启用标准的 KL 散度损失（通常用于限制模型偏离初始模型）。 |
| `loss_agg_mode` | `token-mean` | 损失聚合方式。计算 Loss 时是对所有 Token 取平均，还是对序列取平均。 |



#  数据与长度控制
这部分控制输入输出的文本长度以及如何处理超长数据。

| 参数名 | 值/示例 | 含义解释 |
| :--- | :--- | :--- |
| `max_prompt_length` | `2048` | 最大提示词长度。输入的问题或上下文最长允许多少个 Token。 |
| `max_response_length` | `20480` | 最大回复长度。模型生成的回答最长允许多少个 Token（这里设得很大，适合数学推理等长文本任务）。 |
| `enable_overlong_buffer` | `True` | 是否开启超长缓冲区。用于处理超过最大长度限制的样本，防止直接截断导致信息丢失。 |
| `overlong_penalty_factor` | `1.0` | 超长惩罚系数。对超长回复的奖励进行打折的系数。 |



# 训练批次与采样策略
这部分决定了每次训练“吃”多少数据，以及每个问题生成多少个答案。

| 参数名 | 值/示例 | 含义解释 |
| :--- | :--- | :--- |
| `train_prompt_bsz` | `512` | 训练批次大小。每次更新参数时使用的 Prompt 数量。 |
| `gen_prompt_bsz` | `2048` | 生成批次大小。每次推理生成数据时的 Prompt 数量（通常是训练批次的倍数）。 |
| `n_resp_per_prompt` | `16` | 每个提示词生成的回复数。这是 GRPO 的关键参数，每个问题生成 16 个不同答案组成一个“组”来计算优势。 |
| `train_prompt_mini_bsz` | `64` | 训练微批次大小。在梯度累积或显存受限时，每次实际送入 GPU 计算的样本数。 |
| `enable_filter_groups` | `True` | 是否开启组过滤。根据某种指标（如准确率 `acc`）过滤掉质量差的生成组，不参与训练。 |



# PPO/GRPO 核心超参数
这些是强化学习算法的数学参数，控制更新的幅度和稳定性。

| 参数名 | 值/示例 | 含义解释 |
| :--- | :--- | :--- |
| `clip_ratio_low` | `0.2` | PPO 裁剪下界。限制策略更新幅度的下限（ $ \epsilon $ ）。 |
| `clip_ratio_high` | `0.28` | PPO 裁剪上界。限制策略更新幅度的上限。 |
| `temperature` | `1.0` | 采样温度。生成时的随机性，值越大越随机。 |
| `top_p` | `1.0` | 核采样参数。控制生成时的多样性。 |
| `val_top_p` | `0.7` | 验证时核采样。验证模式下使用更确定的采样策略（0.7）。 |



# 分布式与性能优化
这部分针对多机多卡环境，优化显存占用和计算速度。

| 参数名 | 值/示例 | 含义解释 |
| :--- | :--- | :--- |
| `sp_size` | `8` | 序列并行大小。使用 Ulysses 等序列并行技术，将长序列拆分到 8 张卡上计算。 |
| `gen_tp` | `4` | 生成时张量并行大小。推理（生成）阶段使用 4 张卡进行张量并行。 |
| `offload` | `True` | 参数卸载。将部分模型参数卸载到 CPU 内存，以节省 GPU 显存（速度会变慢，但能跑更大的模型）。 |
| `use_dynamic_bsz` | `True` | 动态批次大小。根据序列长度动态调整批次大小，提高 GPU 利用率。 |



# 路径与环境配置
这部分定义了文件存储位置和运行环境。
- MODEL_PATH: 预训练模型（Qwen2.5-32B）的存放路径。
- TRAIN_FILE / TEST_FILE: 训练数据（如 dapo-math-17k）和测试数据（如 aime-2024）的路径。
- CKPTS_DIR: 训练产生的检查点（Checkpoint）保存目录。
- NNODES: 使用的节点数量（这里设置了 16 个节点）。

# 启动命令参数
脚本最后通过 ray job submit 启动训练，传递了上述所有配置：
- trainer.logger: 指定日志输出方式（控制台 + WandB）。
- trainer.test_freq: 每训练多少步进行一次验证。
- trainer.save_freq: 每训练多少步保存一次模型。
- trainer.total_epochs: 训练的总轮次（这里设为 1）。